In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, classification_report, roc_curve

In [30]:
df1= pd.read_csv("Phishing URL.csv")
df2=pd.read_csv("new_data_urls.csv")
df3=pd.read_csv("legitimate_urls_400.csv")

In [31]:
df1

,url,label
0,https://qrco.de/bfMxGP,1
1,https://proj-ac6e2.web.app/,1
2,http://tdameritrade.com.hk,0
3,https://t.co/JGY0alBNjW,1
4,http://hbmcom.net,0
...,...,...
99031,https://lbaajio.com/webcenter/portal/BanBajio/,1
99032,http://mehr.digital,0
99033,http://addisontx.gov,0
99034,https://s-cas-unicaen-fr.weebly.com/,1


In [32]:
df2

,url,status
0,0000111servicehelpdesk.godaddysites.com,0
1,000011accesswebform.godaddysites.com,0
2,00003.online,0
3,0009servicedeskowa.godaddysites.com,0
4,000n38p.wcomhost.com,0
...,...,...
822005,zzufg.com,0
822006,zzu.li,0
822007,zzz.co.uk,0
822008,zzzoolight.co.za,0


In [33]:
df2["status"]= 1-df2["status"]

In [34]:
df2

,url,status
0,0000111servicehelpdesk.godaddysites.com,1
1,000011accesswebform.godaddysites.com,1
2,00003.online,1
3,0009servicedeskowa.godaddysites.com,1
4,000n38p.wcomhost.com,1
...,...,...
822005,zzufg.com,1
822006,zzu.li,1
822007,zzz.co.uk,1
822008,zzzoolight.co.za,1


In [35]:
df1.rename(columns={"label":"status"}, inplace=True)

In [36]:
df1

,url,status
0,https://qrco.de/bfMxGP,1
1,https://proj-ac6e2.web.app/,1
2,http://tdameritrade.com.hk,0
3,https://t.co/JGY0alBNjW,1
4,http://hbmcom.net,0
...,...,...
99031,https://lbaajio.com/webcenter/portal/BanBajio/,1
99032,http://mehr.digital,0
99033,http://addisontx.gov,0
99034,https://s-cas-unicaen-fr.weebly.com/,1


In [37]:
df=pd.concat([df1, df2, df3], ignore_index=True)

In [38]:
df.shape

(921405, 2)

In [39]:
df.isnull().sum()

url       0
status    0
dtype: int64

In [40]:
df.duplicated().sum()

np.int64(14052)

In [41]:
df=df.drop_duplicates()

In [42]:
df.shape

(907353, 2)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 907353 entries, 0 to 921404
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     907353 non-null  object
 1   status  907353 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 20.8+ MB


In [44]:
df[df["status"]==0]

,url,status
2,http://tdameritrade.com.hk,0
4,http://hbmcom.net,0
9,http://dom.by,0
10,http://gulfcoastmedia.com,0
17,http://pcinfo-web.com,0
...,...,...
921400,https://www.sciencedirect.com,0
921401,https://www.ieee.org,0
921402,https://www.acm.org,0
921403,https://www.researchgate.net,0


In [45]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df["url"])

In [46]:
y = df['status'].values
print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (907353, 892550)


In [47]:
#SPLIT DATA
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [48]:
# Option 1: MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [49]:
# 4. Evaluate
y_pred = model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[88224  7253]
 [ 9696 76298]]

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.92      0.91     95477
           1       0.91      0.89      0.90     85994

    accuracy                           0.91    181471
   macro avg       0.91      0.91      0.91    181471
weighted avg       0.91      0.91      0.91    181471



In [50]:
# Predict from user input
def predict_url(url, model, vectorizer):
    """
    Predict whether a URL is phishing or legitimate.

    Parameters:
        url (str): User input URL
        model: Trained ML model
        vectorizer: Fitted TfidfVectorizer

    Returns:
        dict: prediction result with probabilities
    """
    if not isinstance(url, str) or not url.strip():
        raise ValueError("Input must be a non-empty string.")

    url_transformed = vectorizer.transform([url])
    prediction = model.predict(url_transformed)[0]

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(url_transformed)[0]
        phishing_prob = probabilities[1] * 100
        legit_prob = probabilities[0] * 100
    else:
        phishing_prob = legit_prob = None

    print("\n" + "=" * 50)
    print(f"URL: {url}")
    print("=" * 50)
    print(f"Prediction : {'🚨 Phishing' if prediction == 1 else '✅ Legitimate'}")
    if phishing_prob is not None:
        print(f"Phishing Probability  : {phishing_prob:.2f}%")
        print(f"Legitimate Probability: {legit_prob:.2f}%")
    print("=" * 50)

    return {
        "prediction": int(prediction),
        "phishing_probability": phishing_prob,
        "legitimate_probability": legit_prob
    }

In [51]:
# SINGLE MESSAGE TEST
user_message = input("Enter a message to check: ")
predict_message(user_message, model, vectorizer)

Enter a message to check:  www.facebook.com



Message: www.facebook.com
Prediction : ✅ Not Phishing
Phishing Probability     : 6.75%
Not Phishing Probability : 93.25%


{'prediction': 0,
 'phishing_probability': np.float64(6.75245707052711),
 'not_phishing_probability': np.float64(93.2475429294729)}

In [25]:
df3

,url,status
0,https://www.google.com,0
1,https://www.youtube.com,0
2,https://www.facebook.com,0
3,https://www.instagram.com,0
4,https://www.twitter.com,0
...,...,...
354,https://www.sciencedirect.com,0
355,https://www.ieee.org,0
356,https://www.acm.org,0
357,https://www.researchgate.net,0


In [26]:
# MULTIPLE MESSAGES IN A LOOP
print("\n--- Batch Mode (type 'quit' to exit) ---")
while True:
    user_input = input("\nEnter message: ").strip()
    
    if user_input.lower() == 'quit':
        print("Exiting...")
        break
    
    if not user_input:
        print("Please enter a valid message.")
        continue
        
    predict_message(user_input, model, vectorizer)


--- Batch Mode (type 'quit' to exit) ---



Enter message:  https://www.twitter.com



Message: https://www.twitter.com
Prediction : 🚨 Phising
Phising  Prob : 63.29%
Not Phising   Prob : 36.71%



Enter message:  quit


Exiting...


In [52]:
import pickle

#SAVE MODEL AND VECTORIZER
with open('phising_url.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('phising_url_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [54]:
# LOAD MODEL AND VECTORIZER
with open('spam_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('tfidf_vectorizer.pkl', 'rb') as f:
    loaded_tfidf = pickle.load(f)

print("Model and vectorizer loaded successfully!")

Model and vectorizer loaded successfully!


In [ ]:
# # Option 1: ComplementNB (designed for imbalanced text classification)
# model = ComplementNB()
# model.fit(X_train, y_train)

In [28]:
# # 4. Evaluate
# y_pred = model.predict(X_test)
# print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[87988  7417]
 [ 9368 76626]]

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.92      0.91     95405
           1       0.91      0.89      0.90     85994

    accuracy                           0.91    181399
   macro avg       0.91      0.91      0.91    181399
weighted avg       0.91      0.91      0.91    181399



In [29]:
# # PREDICT FROM USER INPUT
# def predict_message(message, model, vectorizer):
#     # Transform the input using the already-fitted tfidf
#     message_transformed = vectorizer.transform([message])
    
#     # Get prediction and probability
#     prediction = model.predict(message_transformed)[0]
#     probability = model.predict_proba(message_transformed)[0]
    
#     Phising_prob  = probability[1] * 100
#     not_Phising_prob   = probability[0] * 100
    
#     print("\n" + "="*50)
#     print(f"Message: {message}")
#     print(f"{'='*50}")
#     print(f"Prediction : {'🚨 Phising' if prediction == 1 else '✅  Not Phising'}")
#     print(f"Phising  Prob : {Phising_prob:.2f}%")
#     print(f"Not Phising   Prob : {not_Phising_prob:.2f}%")
#     print("="*50)
    
#     return prediction, Phising_prob

In [ ]:
# # SINGLE MESSAGE TEST
# user_message = input("Enter a message to check: ")
# predict_message(user_message, model, vectorizer)

In [ ]:
# # MULTIPLE MESSAGES IN A LOOP
# print("\n--- Batch Mode (type 'quit' to exit) ---")
# while True:
#     user_input = input("\nEnter message: ").strip()
    
#     if user_input.lower() == 'quit':
#         print("Exiting...")
#         break
    
#     if not user_input:
#         print("Please enter a valid message.")
#         continue
        
#     predict_message(user_input, model, vectorizer)